<a href="https://colab.research.google.com/github/AArashinAA/Crypto/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install chromaDB

In [ ]:
import os
import docx
# import pandas as pd
# import numpy as np
import PyPDF2
import chromadb
from chromadb.utils import embedding_functions

In [ ]:
def read_text_file(filePath):
    with open(filePath, 'r', encoding='utf-8') as f:
        return f.read()

In [ ]:
def read_pdf_file(filePath):
    with open(filePath, 'rb') as f:
        pdfReader = PyPDF2.PdfReader(f)
        text = ''
        for page in pdfReader.pages:
            text += page.extract_text() + '\n'
        return text

In [ ]:
def read_docx_file(file_path):
    doc = docx.Document(file_path)
    text = '\n'.join([paragraph.text for paragraph in doc.paragraphs])
    return text

In [ ]:
def read_document(filePath):
    if filePath.endswith('.txt'):
        return read_text_file(filePath)
    elif filePath.endswith('.pdf'):
        return read_pdf_file(filePath)
    elif filePath.endswith('.docx'):
        return read_docx_file(filePath)

In [ ]:
def split_text(text, chunk_size=500, chunk_overlap=100):
    # splitting text into overlapping chunks
    text = text.replace('\n', ' ')
    start = 0
    length = len(text)
    chunks = []
    while start < length:
        end = min(start + chunk_size, length)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - chunk_overlap

    return chunks

In [ ]:
file_path = './drylab.pdf'
text = read_document(file_path)
print(text[:500])

Drylab Newsfor investors & friends · May 2 017
Welcome to our first newsletter of 2017! It's
been a while since the last one, and a lot has
happened. W e promise to k eep them coming
every two months hereafter , and permit
ourselv es to mak e this one r ather long. The
big news is the beginnings of our launch in
the American mark et, but there are also
interesting updates on sales, de velopment,
mentors and ( of course ) the in vestment
round that closed in January .
New capital: The in vestment


In [ ]:
chunks = split_text(text)

In [84]:
words = chunks[0].split()
print(len(words))

98


In [ ]:
client = chromadb.PersistentClient(path="./chorama_db")
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2")
collection = client.get_or_create_collection(
    name="my_collection",
    embedding_function=sentence_transformer_ef)

In [ ]:
def process_document(file_path):
    text = read_document(file_path)
    chunks = split_text(text)
    file_name = os.path.basename(file_path)
    metadatas = [{'source': file_name, 'chunk_index': i} for i in range(len(chunks))]
    ids = [f"{file_name}_chunks_{i}" for i in range(len(chunks))]
    return ids, chunks, metadatas


In [ ]:
ids, chunks, metadatas = process_document(file_path)

In [ ]:
print(ids[:5])

['drylab.pdf_chunks_0', 'drylab.pdf_chunks_1', 'drylab.pdf_chunks_2', 'drylab.pdf_chunks_3', 'drylab.pdf_chunks_4']


In [ ]:
collection.add(
    documents=chunks,
    metadatas=metadatas,
    ids=ids
)

In [ ]:
def semantic_search(query, k=2):
    results = collection.query(
        query_texts=[query],
        n_results=k,
        include=["documents", "metadatas"]
    )
    return results

def get_context_with_source(semantic_search_result):
    if not semantic_search_result or not semantic_search_result.get("documents") or not semantic_search_result["documents"][0]:
        return "", []
    context = "\n\n".join(semantic_search_result["documents"][0])
    seen = set()
    sources = []
    for metadata in semantic_search_result["metadatas"][0]:
        source = f"{metadata.get("source","?")}: {metadata.get('chunk_index', '?')}"
        if source and source not in seen:
            sources.append(source)
            seen.add(source)
    return context, sources


In [ ]:
def ask(quey, n_results):
  semantic_search_result = semantic_search(quey, k=n_results)
  context, sources = get_context_with_source(semantic_search_result)
  print("=========== Context ===========")
  print(context or "no match found")
  print("=========== Sources ===========")
  if sources:
    for i, s in enumerate(sources, 1):
      print(f"{i}. {s}")
    else:
      print("no source")
  return context, sources


In [42]:
quert = "what is drylab?"
context, sources = ask(quert, 2)

=========== Context ===========
Drylab Newsfor investors & friends · May 2 017 Welcome to our first newsletter of 2017! It's been a while since the last one, and a lot has happened. W e promise to k eep them coming every two months hereafter , and permit ourselv es to mak e this one r ather long. The big news is the beginnings of our launch in the American mark et, but there are also interesting updates on sales, de velopment, mentors and ( of course ) the in vestment round that closed in January . New capital: The in vestment

expect the dela y to decrease, with new de velopers on board.The launch of Drylab 3.0 will tak e place at the International Broadcasters Con vention in Amsterdam in September , and we are working hard to get solid feedback from pilot users before then. Annual General Meeting: Drylab 's A GM will be held on June 16th at 15:00. An invitation will be distributed to all owners well in advance. W e hope to see y ou there! As you can see it has been a hectic spring th

In [72]:
OPEN_ROUTER_AI = "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"
OPEN_ROUTER_MODEL_NAME = "openai/gpt-oss-20b:free"

In [73]:
from openai import OpenAI

In [74]:
client = OpenAI(
  api_key=OPEN_ROUTER_AI,
  base_url="https://openrouter.ai/api/v1"
)

In [77]:
system_prompt = ("You are a helpful assistant for retrieval-augmented generation (RAG).\n",
        "Answer ONLY using the provided context.\n",
        "If the answer is not found in the context, say:",
        "'I don't know based on the provided documents.'",)

In [76]:
def build_message(context, question):
  return [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}\nAnswer:"},
  ]

In [78]:
def rag_answer(query, n_results):
  semantic_search_result = semantic_search(query, k=n_results)
  context, sources = get_context_with_source(semantic_search_result)
  if not context:
    return "I don't know based on the provided documents.", []
  message = build_message(context, query)
  response = client.chat.completions.create(
    model=OPEN_ROUTER_MODEL_NAME,
    messages=message,
    temperature=0.2,
    max_tokens=512
  )

  answer = response.choices[0].message.content

  print("=========== Answer ===========")
  if answer:
    for i, s in enumerate(sources, 1):
      print(f"{i}. {s}")
    else:
      print("no source")
  return answer, sources

In [79]:
query = "what is drylab?"
context, sources = rag_answer(query, 2)

RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'openai/gpt-oss-20b:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Chutes'}}, 'user_id': 'user_37ndyelju12FXf4dqIkhnJ6ADE2'}